# Movie Recommendation ML Pipeline

Notebook ini dibuat untuk presentasi. Fokusnya bukan eksperimen panjang, tetapi menjelaskan alur kerja project dari data film sampai rekomendasi menggunakan Qdrant.

Tujuan akhir project:

```text
User input judul film -> sistem mencari film mirip -> chatbot menampilkan rekomendasi
```

## 1. Ringkasan Jawaban Jika Ditanya Dosen

**Jenis sistem rekomendasi:** content-based recommendation.

**Alasan:** dataset tidak punya histori user, rating personal, atau click history. Jadi rekomendasi dibuat berdasarkan kemiripan konten film.

**Model yang digunakan:** PySpark ML Word2Vec.

**Fungsi model:** mengubah deskripsi/metadata film menjadi vector angka berukuran 64 dimensi.

**Vector database:** Qdrant.

**Fungsi Qdrant:** menyimpan vector film dan mencari film lain yang vector-nya paling mirip.

**Similarity:** cosine similarity di Qdrant.

**Library utama:** pandas, PySpark, pyarrow, requests, Qdrant, Docker.

## 2. Alur Pengerjaan

```text
1. Scraping TMDB
   Ambil data film dari TMDB.

2. Data enrichment
   Tambahkan detail seperti director, cast, keywords, runtime, budget, revenue.

3. EDA
   Pahami kualitas data, missing values, distribusi genre, rating, tahun, dan sebagainya.

4. Preprocessing
   Bersihkan data, rapikan teks, buang data yang tidak layak.

5. Feature engineering
   Gabungkan metadata penting menjadi movie_document_weighted.

6. Finalisasi dataset
   Buat movies_final.csv dan movies_payload.csv.

7. Training Word2Vec
   Ubah setiap film menjadi vector angka.

8. Indexing Qdrant
   Masukkan vector dan metadata film ke collection movies.

9. Search rekomendasi
   Cari film dengan vector paling mirip.
```

## 3. Fungsi Folder Project

| Folder | Fungsi |
| --- | --- |
| `docs/` | Dokumentasi, progress, task breakdown, dan setup Docker. |
| `ml/data/` | Dataset lokal. Tidak di-push karena besar. |
| `ml/notebooks/` | Notebook eksperimen dan presentasi. |
| `ml/scripts/` | Pipeline final yang bisa dijalankan ulang. |
| `ml/reports/` | Ringkasan, grafik, dan hasil report kecil. |
| `ml/models/` | Model hasil training. Tidak di-push karena bisa dibuat ulang. |
| `qdrant_storage/` | Data internal Qdrant lokal. Tidak di-push. |
| `backend/` | Nanti untuk FastAPI. Belum dibuat. |
| `frontend/` | Nanti untuk chatbot UI. Belum dibuat. |

## 4. Library Yang Digunakan

| Library | Dipakai Untuk |
| --- | --- |
| `pandas` | Membaca dan membersihkan CSV, membuat dataset final, membaca payload. |
| `PySpark` | Training Word2Vec dan pemrosesan data berbasis Spark. |
| `pyarrow` | Membaca/menulis file Parquet untuk vector film. |
| `requests` | Memanggil API TMDB dan API Qdrant. |
| `python-dotenv` | Membaca konfigurasi dari `.env`. |
| `Qdrant` | Vector database untuk similarity search. |
| `Docker` | Menyamakan environment agar bisa dijalankan oleh tim. |

In [ ]:
from pathlib import Path
import os
import pandas as pd
import requests

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'ml').exists() and (candidate / 'docker-compose.yml').exists():
            return candidate
    raise RuntimeError('Project root tidak ditemukan')

PROJECT_ROOT = find_project_root(Path.cwd())
ML_DIR = PROJECT_ROOT / 'ml'
DATA_DIR = ML_DIR / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
FINAL_DATA_DIR = PROCESSED_DIR / 'final'
VECTOR_DATA_DIR = PROCESSED_DIR / 'vectors'
REPORT_DIR = ML_DIR / 'reports'

PROJECT_ROOT

## 5. Dataset Final

Dataset final dibuat oleh script:

```bash
python ml/scripts/preprocessing/finalize_datasets.py
```

Output penting:

- `movies_final.csv`: dataset utama untuk modelling.
- `movies_payload.csv`: metadata ringkas untuk backend dan Qdrant.
- `data_summary.csv`: ringkasan kualitas data final.

In [ ]:
data_summary_path = REPORT_DIR / 'data' / 'data_summary.csv'
data_summary = pd.read_csv(data_summary_path)
data_summary

In [ ]:
payload_path = FINAL_DATA_DIR / 'movies_payload.csv'
payload_preview = pd.read_csv(payload_path, nrows=5)
payload_preview

## 6. Feature Utama: movie_document_weighted

`movie_document_weighted` adalah gabungan metadata penting film, misalnya:

- title
- genre
- director
- cast
- keywords
- overview
- tagline

Tujuannya adalah membuat satu teks representasi film. Teks ini kemudian dipakai untuk training Word2Vec.

In [ ]:
movies_final_path = FINAL_DATA_DIR / 'movies_final.csv'
sample_cols = ['id', 'title', 'genres_text', 'director_clean', 'movie_document_weighted']
sample_movies = pd.read_csv(movies_final_path, usecols=sample_cols, nrows=3)
sample_movies

## 7. Model: PySpark Word2Vec

Word2Vec membaca token/kata dari `movie_document_weighted`, lalu membuat vector untuk setiap film.

Alasan menggunakan Word2Vec:

- cocok untuk teks metadata film,
- menghasilkan dense vector,
- dense vector cocok dimasukkan ke Qdrant,
- sesuai konteks Big Data karena menggunakan PySpark.

Script training:

```bash
python ml/scripts/modelling/train_word2vec.py
```

In [ ]:
word2vec_summary_path = REPORT_DIR / 'modelling' / 'training' / 'word2vec_summary.csv'
word2vec_summary = pd.read_csv(word2vec_summary_path)
word2vec_summary

In [ ]:
vector_path = VECTOR_DATA_DIR / 'movie_vectors_word2vec.parquet'
vectors_preview = pd.read_parquet(vector_path).head(3)
vectors_preview[['id', 'title', 'release_year', 'genres_text', 'vector']]

## 8. Qdrant Vector Database

Qdrant menyimpan vector film dan melakukan similarity search.

Script indexing:

```bash
python ml/scripts/indexing/index_qdrant.py
```

Collection yang dibuat:

```text
movies
```

Distance yang digunakan:

```text
Cosine
```

In [ ]:
QDRANT_HOST = os.getenv('QDRANT_HOST', 'localhost')
QDRANT_PORT = int(os.getenv('QDRANT_PORT', '6333'))
QDRANT_COLLECTION = os.getenv('QDRANT_COLLECTION', 'movies')
QDRANT_URL = f'http://{QDRANT_HOST}:{QDRANT_PORT}'

response = requests.get(f'{QDRANT_URL}/collections/{QDRANT_COLLECTION}', timeout=30)
response.status_code, response.json().get('result', {}).get('status'), response.json().get('result', {}).get('points_count')

## 9. Demo Rekomendasi Dari Qdrant

Contoh input:

```text
Saya ingin nonton film seperti Interstellar
```

Untuk MVP, sistem mengambil judul film acuan `Interstellar`, lalu mencari film dengan vector paling mirip di Qdrant.

In [ ]:
def find_movie_by_title(title: str) -> pd.Series:
    payload = pd.read_csv(payload_path, usecols=['id', 'title', 'release_year'])
    payload['id'] = pd.to_numeric(payload['id'], errors='coerce').astype('int64')
    titles = payload['title'].fillna('')
    exact = payload[titles.str.lower() == title.lower()].sort_values(['title', 'release_year'])
    fuzzy = payload[titles.str.contains(title, case=False, regex=False)].sort_values(['title', 'release_year'])
    candidates = exact if not exact.empty else fuzzy
    if candidates.empty:
        raise ValueError(f'Tidak ada film yang cocok dengan title: {title}')
    return candidates.iloc[0]

def recommend_from_qdrant(title: str, top_k: int = 10) -> pd.DataFrame:
    source = find_movie_by_title(title)
    source_id = int(source['id'])
    vectors = pd.read_parquet(vector_path, columns=['id', 'vector'])
    vectors['id'] = pd.to_numeric(vectors['id'], errors='coerce').astype('int64')
    source_vector = vectors.loc[vectors['id'] == source_id, 'vector']
    if source_vector.empty:
        raise ValueError(f'Vector tidak ditemukan untuk id: {source_id}')

    search_response = requests.post(
        f'{QDRANT_URL}/collections/{QDRANT_COLLECTION}/points/search',
        json={
            'vector': [float(value) for value in source_vector.iloc[0]],
            'limit': top_k + 1,
            'with_payload': True,
        },
        timeout=120,
    )
    search_response.raise_for_status()

    rows = []
    for result in search_response.json().get('result', []):
        if int(result['id']) == source_id:
            continue
        movie = result.get('payload', {})
        rows.append({
            'score': result.get('score'),
            'title': movie.get('title'),
            'release_year': movie.get('release_year'),
            'genres_text': movie.get('genres_text'),
            'vote_average': movie.get('vote_average'),
        })
        if len(rows) >= top_k:
            break

    print(f"Source movie: {source['title']} ({int(source['release_year'])})")
    return pd.DataFrame(rows)

recommend_from_qdrant('Interstellar', top_k=10)

## 10. Jika Input User Lebih Natural

Contoh:

```text
Saya ingin nonton film seperti batman
```

Langkah yang dilakukan sistem:

```text
1. Ambil keyword/judul penting: batman
2. Cari kandidat judul yang mengandung batman
3. Jika hasilnya banyak, minta user memilih film acuan
4. Ambil vector film acuan
5. Cari film mirip di Qdrant
```

In [ ]:
payload = pd.read_csv(payload_path, usecols=['id', 'title', 'release_year', 'genres_text'])
payload[payload['title'].fillna('').str.contains('batman', case=False, regex=False)].head(10)

## 11. Command Untuk Menjalankan Pipeline

Jika ingin menjalankan ulang dari awal, gunakan Docker:

```bash
docker compose up -d qdrant
docker compose run --rm ml python ml/scripts/preprocessing/finalize_datasets.py
docker compose run --rm ml python ml/scripts/modelling/train_word2vec.py
docker compose run --rm ml python ml/scripts/indexing/index_qdrant.py
docker compose run --rm ml python ml/scripts/indexing/test_qdrant_search.py Interstellar 10
```

## 12. Kesimpulan Presentasi

Project ini menggunakan pendekatan content-based recommendation karena tidak ada data perilaku user.

Metadata film digabung menjadi `movie_document_weighted`, lalu dilatih menggunakan PySpark Word2Vec untuk menghasilkan vector film. Vector tersebut disimpan ke Qdrant agar sistem bisa melakukan similarity search dengan cepat.

Pipeline utama:

```text
TMDB data -> preprocessing -> movie_document_weighted -> Word2Vec -> vector -> Qdrant -> recommendation
```